# Matched filter and shot averaging for the absorption lock-in

This notebook measures the cloud's absorption signal with **less noise** than the single-shot lock-in notebooks, then averages many shots. It changes **only the analysis**; the data are the same.

**The idea in one paragraph.** The lock-in turns the photodiode signal into a slowly varying complex number Z(t). It then smooths Z with a short filter, about 0.5 ms, so that fast features stay visible. But the cloud takes much longer than that to cross the probe: the PMT shows a Gaussian transit with σ ≈ 1.75 ms on the way up and σ ≈ 5.8 ms on the way back. A filter much shorter than the signal lets through far more noise than it needs to. The best linear filter for a pulse of known shape in noise is one with the **same shape as the pulse**: the *matched filter*. Here that is a Gaussian whose width equals the transit width, measured independently by the PMT.

**What the notebook does, step by step:**

1. Load the selected shots, skipping clipped, no-MOT and no-launch shots, plus a set of no-launch shots for measuring the noise.
2. Fit each shot's PMT fluorescence to find when the cloud is centred in the probe (t_c) and its transit width σ.
3. Remove the constant lock-in background, then find the signal phase common to all shots.
4. Measure the true noise of each filter on the no-launch shots. Wide filters cannot measure their own noise inside the 70 ms window.
5. Compare the per-shot SNR of the current narrow filter and the matched filter.
6. Scan the filter width to check that the best width really is the transit width.
7. Average shots, aligned on each shot's own PMT time, and see how the SNR grows with the number of shots.

**SNR definition used here.** Each shot's signal is read at the PMT-defined time t_c, projected onto the common signal phase. It is divided by the measured standard deviation of the same quantity in pure noise. Pure noise therefore gives values centred on 0 with spread 1, like a Gaussian z-score.

This differs from the single-shot notebooks, which take the *largest* |Z| near the cloud and divide by the RMS of both quadratures. That definition gives about 2.4 for pure noise. The two are not directly comparable, so Step 5 shows both.

The calculations live in `cloud_signal_estimation.py`, next to this notebook; every function there is commented. **To use:** edit the inputs in the next cell, then **Restart Kernel and Run All**.

In [ ]:
# Inputs, imports, and the plotting style shared with the other notebooks.
import sys
import importlib
import warnings
from pathlib import Path
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# ---- CHANGE THESE INPUTS --------------------------------------------------------------------
DAY_DIR = Path(r"C:\Experiments\eedmLabscriptOps\exp12_switchArmsAndLaunch_Absorption\2026\10\14")
# Shot folders with the atoms launched. A runmanager sequence is one folder with many files.
SIGNAL_FOLDERS = list(range(49, 59)) + [127, 129]
# Only shots whose globals match these values are used (here: the chosen operating point).
SIGNAL_SELECT = {"mod_offset": 0.006, "mod_amp": 0.010}
# No-launch shots (MRR_TRIG_BOOL = 0) used to measure the real noise of each filter.
NOISE_FOLDERS = list(range(59, 69)) + [69]
# Filter widths (s) for the width scan in Step 6.
WIDTH_SCAN_S = [0.1e-3, 0.25e-3, 0.5e-3, 1e-3, 2e-3, 3e-3, 4e-3, 6e-3, 8e-3]
# ----------------------------------------------------------------------------------------------

CLIP_V = 9.95              # PDA100A2 output limit; any sample at or above this is clipped
MIN_MOT_ATOMS_MV = 10.0    # a loaded MOT gives ~42 mV; repump dropouts give < 1 mV
RETURN_DELAY_S = 0.2       # t_PMT_delay above this is the returning pass
PASS_NAMES = {"0": "upward", "1": "returning"}

import fluorescence_absorption_lockin as lockin_helper
import cloud_signal_estimation as cse
importlib.reload(lockin_helper)
importlib.reload(cse)
from fluorescence_absorption_lockin import (load_shot, fit_modulation_reference, analyze_shot,
                                            load_mot_trace, fit_mot_loading)

SAVE_OUTPUTS = True
OUTPUT_DIR = DAY_DIR / "matched_filter_outputs" / f"{SIGNAL_FOLDERS[0]:04d}-{SIGNAL_FOLDERS[-1]:04d}"
if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings("ignore", category=RuntimeWarning)
%matplotlib widget
plt.ioff()
plt.rcParams.update({"figure.dpi": 110, "font.size": 12, "axes.grid": True, "grid.alpha": 0.25,
                     "lines.linewidth": 1.0, "axes.labelsize": 12, "axes.titlesize": 12,
                     "legend.fontsize": 11, "xtick.labelsize": 12, "ytick.labelsize": 12})
label_font = {"fontname": "sans-serif", "size": 12, "color": "black", "weight": "normal",
              "verticalalignment": "bottom"}
title_font = {"fontname": "sans-serif", "size": 12, "color": "black", "weight": "normal"}
legend_font = {"family": "sans-serif", "size": 11, "style": "normal", "weight": "normal"}
EXPORT_SIZES_INCHES = {"timing": (14, 8), "phase": (14, 7), "noise_check": (14, 6),
                       "per_shot_snr": (14, 8), "width_scan": (14, 6), "averaging": (16, 11)}

def show_figure(fig, name):
    """Export at a fixed size and display the live, resizable widget canvas."""
    if SAVE_OUTPUTS:
        display_size = fig.get_size_inches().copy()
        try:
            fig.set_size_inches(EXPORT_SIZES_INCHES[name], forward=False)
            fig.savefig(OUTPUT_DIR / f"{name}.png", dpi=180, bbox_inches="tight")
        finally:
            fig.set_size_inches(display_size, forward=False)
    fig.canvas.layout.width = "100%"
    fig.canvas.layout.height = "auto"
    fig.canvas.resizable = True
    fig.canvas.toolbar_visible = True
    fig.canvas.toolbar_position = "top"
    fig.canvas.header_visible = False
    display(fig.canvas)

## Step 1 · Load and select shots

Every `.h5` file in the listed folders is opened read-only. A **signal** shot is kept only if:

- its globals match `SIGNAL_SELECT`;
- it was launched (`MRR_TRIG_BOOL = 1`);
- it has a MOT (atom signal above 10 mV);
- the photodiode never reached its 10 V output limit (clipping distorts the lock-in).

**Noise** shots must be no-launch shots that are not clipped. The modulation reference is fitted once per shot. Its phase θ(t) is what the lock-in mixes with.

In [ ]:
def numeric_globals(path):
    """Scalar numeric globals of a shot (runmanager settings)."""
    with h5py.File(path, "r") as h5:
        return {k: float(v) for k, v in h5["globals"].attrs.items()
                if np.ndim(v) == 0 and isinstance(v, (int, float, np.integer, np.floating, np.bool_))}

def load_set(folders, select, want_launch):
    """Load all usable shots from the folders; count why the others were skipped."""
    kept, skipped = [], {}
    for path in (p for n in folders for p in sorted((DAY_DIR / f"{n:04d}").glob("*.h5"))):
        with h5py.File(path, "r") as h5:
            ran = "data/traces" in h5
        g = numeric_globals(path)
        reason = None
        if not ran:
            reason = "not run"
        elif any(abs(g.get(k, np.nan) - v) > 1e-9 for k, v in select.items()):
            reason = "different settings"
        elif bool(g.get("MRR_TRIG_BOOL", 1)) != want_launch:
            reason = "launch flag"
        if reason is None:
            shot = load_shot(path)
            mot = load_mot_trace(path)
            if shot["pda_v"].max() >= CLIP_V:
                reason = "clipped"
            elif want_launch and mot is not None and \
                    1e3 * fit_mot_loading(mot)["atoms_v_at_coils_off"] < MIN_MOT_ATOMS_MV:
                reason = "no MOT"
        if reason is not None:
            skipped[reason] = skipped.get(reason, 0) + 1
            continue
        reference = fit_modulation_reference(shot, frequency_guess_hz=shot["mod_freq"] or 6123.0)
        kept.append({"file": path.name, "folder": path.parent.name, "shot": shot, "ref": reference,
                     "acq": "1" if shot["pmt_delay_s"] > RETURN_DELAY_S else "0"})
    return kept, skipped

signal_shots, signal_skipped = load_set(SIGNAL_FOLDERS, SIGNAL_SELECT, want_launch=True)
noise_shots, noise_skipped = load_set(NOISE_FOLDERS, {}, want_launch=False)
counts = pd.DataFrame({
    "signal shots kept": pd.Series([d["acq"] for d in signal_shots]).value_counts(),
    "noise shots kept": pd.Series([d["acq"] for d in noise_shots]).value_counts(),
}).rename(index=PASS_NAMES).fillna(0).astype(int)
display(counts)
print("signal shots skipped:", signal_skipped)
print("noise shots skipped: ", noise_skipped)

## Step 2 · When does the cloud cross the probe, and for how long?

The PMT records fluorescence from atoms inside the probe sheet. Fitting a Gaussian to its smoothed count rate gives two things for every shot, **without using the photodiode at all**:

- **t_c**, the time the cloud is centred in the probe. This is where each shot's signal is read, and what the shots are aligned on.
- **σ**, the transit width. The median σ of each pass becomes that pass's matched-filter width.

Using the PMT keeps the analysis honest. If the photodiode trace chose its own time and width, noise could be mistaken for signal.

In [ ]:
# Fit the PMT envelope of every shot (signal and noise shots alike).
for d in signal_shots + noise_shots:
    d["pmt"] = cse.pmt_cloud_fit(d["shot"], d["ref"]["f_hz"])

timing_rows = []
for d in signal_shots:
    timing_rows.append({"file": d["file"], "pass": PASS_NAMES[d["acq"]], "PMT fit ok": d["pmt"]["ok"],
                        "t_c after drop (ms)": 1e3 * (d["pmt"]["center_s"] + d["shot"]["acquisition_after_drop_s"]),
                        "transit sigma (ms)": 1e3 * d["pmt"]["sigma_s"],
                        "PMT peak (kcps)": d["pmt"]["amp_cps"] / 1e3})
timing = pd.DataFrame(timing_rows)
bad_fits = timing[~timing["PMT fit ok"]]
if not bad_fits.empty:
    display(Markdown(f"**{len(bad_fits)} signal shots have no usable PMT fit and are dropped.**"))
    display(bad_fits)
signal_shots = [d for d in signal_shots if d["pmt"]["ok"]]

# One matched-filter width per pass: the median transit sigma of that pass.
SIGMA_MATCH = {acq: float(np.median([d["pmt"]["sigma_s"] for d in signal_shots if d["acq"] == acq]))
               for acq in sorted({d["acq"] for d in signal_shots})}
# Reference cloud time per pass, used only for no-launch shots (which have no cloud of their own).
T_C_TYPICAL = {acq: float(np.median([d["pmt"]["center_s"] for d in signal_shots if d["acq"] == acq]))
               for acq in SIGMA_MATCH}
display(timing[timing["PMT fit ok"]].groupby("pass")[["t_c after drop (ms)", "transit sigma (ms)", "PMT peak (kcps)"]]
        .agg(["mean", "std", "min", "max"]).round(2))
display(Markdown("**Matched-filter widths:** " + ", ".join(
    f"{PASS_NAMES[a]} σ = {1e3 * s:.2f} ms" for a, s in SIGMA_MATCH.items())))

fig, axes = plt.subplots(2, 2, layout="constrained")
for col, acq in enumerate(SIGMA_MATCH):
    group = timing[(timing["pass"] == PASS_NAMES[acq]) & timing["PMT fit ok"]]
    axes[0, col].plot(range(len(group)), group["t_c after drop (ms)"], "o")
    axes[0, col].set_title(f"{PASS_NAMES[acq]}: cloud centre time", fontdict=title_font)
    axes[0, col].set_ylabel("t_c after drop (ms)", fontdict=label_font, labelpad=12)
    axes[1, col].plot(range(len(group)), group["transit sigma (ms)"], "o", color="C1")
    axes[1, col].axhline(1e3 * SIGMA_MATCH[acq], color="C3", ls="--", label="matched-filter width")
    axes[1, col].set_ylabel("transit σ (ms)", fontdict=label_font, labelpad=12)
    axes[1, col].set_xlabel("shot (in loading order)", fontdict=label_font, labelpad=20)
    axes[1, col].legend(prop=legend_font)
show_figure(fig, "timing")

## Step 3 · Remove the background, then find the common signal phase

Even without atoms, the lock-in output is not zero. Modulating the laser frequency slightly modulates its power too (residual amplitude modulation), and there is electronic pickup at f_m. Both add a **constant complex offset** to Z.

For each shot, that offset is measured with the narrow filter more than 20 ms from the cloud, then subtracted. The 2026-10-14 no-launch tests showed this background is constant: it does not change shape from shot to shot.

The absorption signal itself has a fixed **phase** set by the detuning and the electronics. Adding the complex signal values of all shots at their t_c gives a vector pointing along that phase. Strong shots dominate the sum, and near-zero noisy shots barely move it. Each shot is then **projected** onto this direction, giving a signed amplitude X_signal. The perpendicular component is pure noise and is dropped, which is about √2 better than using |Z|.

In [ ]:
# Filter every signal shot twice (narrow lock-in filter and matched filter) and read both at t_c.
for d in signal_shots:
    d["tr"] = cse.shot_traces(d["shot"], d["ref"], d["pmt"]["center_s"], SIGMA_MATCH[d["acq"]])
    d["z_match_tc"] = cse.value_at(d["tr"]["t"], d["tr"]["z_match"], d["pmt"]["center_s"])
    d["z_narrow_tc"] = cse.value_at(d["tr"]["t"], d["tr"]["z_narrow"], d["pmt"]["center_s"])

PHASE = {acq: cse.common_phase([d["z_match_tc"] for d in signal_shots if d["acq"] == acq])
         for acq in SIGMA_MATCH}
display(Markdown("**Common signal phase:** " + ", ".join(
    f"{PASS_NAMES[a]} {np.rad2deg(p):.1f}°" for a, p in PHASE.items())))

# Plot each shot's matched-filter value in the complex plane: they should cluster along one line.
fig, axes = plt.subplots(1, len(PHASE), layout="constrained", squeeze=False)
for ax, acq in zip(axes[0], PHASE):
    z = np.array([d["z_match_tc"] for d in signal_shots if d["acq"] == acq])
    ax.plot(1e3 * z.real, 1e3 * z.imag, "o", alpha=0.7, label="shots at t_c")
    reach = 1.2e3 * np.abs(z).max()
    ax.plot([0, reach * np.cos(PHASE[acq])], [0, reach * np.sin(PHASE[acq])], color="C3",
            label="common phase")
    ax.axhline(0, color="0.5", lw=0.6)
    ax.axvline(0, color="0.5", lw=0.6)
    ax.set_aspect("equal", adjustable="datalim")
    ax.set_title(f"{PASS_NAMES[acq]}: matched-filter Z at t_c", fontdict=title_font)
    ax.set_xlabel("X (mV)", fontdict=label_font, labelpad=20)
    ax.set_ylabel("Y (mV)", fontdict=label_font, labelpad=12)
    ax.legend(prop=legend_font)
show_figure(fig, "phase")

## Step 4 · Measure the real noise of each filter on no-launch shots

**What "noise" means here.** Each shot contributes **one number**: the filtered signal at the PMT time t_c, projected onto the signal phase. Its noise is the standard deviation that this single number would have in shots without atoms.

Neighbouring filtered samples are strongly correlated, but that does not affect this definition, because only one pre-chosen sample per shot is used: there is no search over samples. A linear filter applied to Gaussian noise also gives Gaussian output, so the SNR behaves like a z-score.

**Prediction.** A wide filter cannot measure its own noise inside the same shot. With σ ≈ 6 ms, it needs about ±18 ms of data on each side, and that leaves no cloud-free stretch in the 70 ms window. Each shot therefore gets a **predicted** matched-filter noise:

1. measure the noise of the narrow filter far from the cloud;
2. scale it by √(Σk_wide² / Σk_narrow²), which is exact if the noise is *white* near f_m.

**Calibration.** The noise is not perfectly white: there is extra slow noise. So the prediction is **calibrated** by applying the same filters to **no-launch shots**, where there is no signal anywhere. The filtered output is read at many positions and divided by its predicted noise. If the prediction were exact, those ratios would have a standard deviation of 1; the measured value, κ, is the correction factor, and all SNRs below are divided by it.

**Correlated readings.** Readings within one shot are correlated. For Gaussian-filtered white noise, the correlation at spacing d is exp(−d²/4σ²): 0.78 one filter width apart, 0.37 at two, 0.11 at three. The number of readings therefore overstates how well κ is known.

The table reports a **68% interval for κ from resampling whole shots** (bootstrap over shots), which accounts for that correlation. On 2026-10-14 this gave about ±4% on the upward pass (37 shots) but **±15% on the return pass** (10 shots, about 2–3 independent readings each). The return-pass SNRs inherit that uncertainty. More no-launch return shots, or a longer modulation window, tighten it.

In [ ]:
def calibrate_noise(acq, width_s=None, narrow=False, return_per_shot=False):
    """Return (kappa, n_readings, z-scores): the spread of noise-only readings in units of the
    predicted noise, for the matched filter of width width_s or for the narrow lock-in filter.
    With return_per_shot=True the z-scores are returned as one array per shot (for the bootstrap)."""
    per_shot = []
    for d in (n for n in noise_shots if n["acq"] == acq):
        tr = cse.shot_traces(d["shot"], d["ref"], T_C_TYPICAL[acq], width_s or SIGMA_MATCH[acq])
        t = tr["t"]
        if narrow:
            positions = t[tr["background"]][::max(1, int(0.002 * d["shot"]["fs_pda"]))]
            trace, predicted = tr["z_narrow"], tr["noise_narrow"]
        else:
            usable = t[tr["valid_match"]]
            if usable.size == 0:
                continue
            # Readings one filter width apart are still correlated (0.78); the bootstrap below
            # accounts for that, so using all of them costs nothing and helps a little.
            positions = np.arange(usable[0], usable[-1], max(width_s or SIGMA_MATCH[acq], 1e-3))
            trace, predicted = tr["z_match"], tr["noise_match"]
        per_shot.append(np.array([cse.project(cse.value_at(t, trace, x), PHASE[acq]) / predicted
                                  for x in positions]))
    z_scores = np.concatenate(per_shot) if per_shot else np.array([])
    kappa = float(np.std(z_scores)) if z_scores.size > 5 else np.nan
    if return_per_shot:
        return kappa, z_scores.size, per_shot
    return kappa, z_scores.size, z_scores

def bootstrap_interval(per_shot, n_boot=4000, seed=0):
    """68% interval of kappa from resampling whole shots, which respects the correlation
    between readings taken from the same shot."""
    rng = np.random.default_rng(seed)
    boot = [np.std(np.concatenate([per_shot[i] for i in rng.integers(0, len(per_shot), len(per_shot))]))
            for _ in range(n_boot)]
    return tuple(np.percentile(boot, [16, 84]))

KAPPA = {}
rows = []
fig, axes = plt.subplots(1, len(PHASE), layout="constrained", squeeze=False)
for ax, acq in zip(axes[0], PHASE):
    k_narrow, n_narrow, z_narrow_shots = calibrate_noise(acq, narrow=True, return_per_shot=True)
    k_match, n_match, z_match_shots = calibrate_noise(acq, return_per_shot=True)
    lo, hi = bootstrap_interval(z_match_shots)
    KAPPA[acq] = {"narrow": k_narrow, "match": k_match, "match_68": (lo, hi)}
    z_narrow, z_match = np.concatenate(z_narrow_shots), np.concatenate(z_match_shots)
    rows.append({"pass": PASS_NAMES[acq], "narrow κ": k_narrow, "narrow readings": n_narrow,
                 "matched κ": k_match, "matched κ 68% low": lo, "matched κ 68% high": hi,
                 "matched readings": n_match, "no-launch shots": len(z_match_shots),
                 "matched mean (should be ≈0)": float(np.mean(z_match))})
    bins = np.linspace(-5, 5, 31)
    ax.hist(z_narrow, bins=bins, alpha=0.5, density=True, label=f"narrow (κ = {k_narrow:.2f})")
    ax.hist(z_match, bins=bins, alpha=0.5, density=True,
            label=f"matched (κ = {k_match:.2f}, 68%: {lo:.2f}–{hi:.2f})")
    x = np.linspace(-5, 5, 200)
    ax.plot(x, np.exp(-x ** 2 / 2) / np.sqrt(2 * np.pi), color="k", label="white-noise prediction")
    ax.set_title(f"{PASS_NAMES[acq]}: noise-only readings / predicted noise", fontdict=title_font)
    ax.set_xlabel("z-score", fontdict=label_font, labelpad=20)
    ax.legend(prop=legend_font)
display(pd.DataFrame(rows).set_index("pass").round(3))
display(Markdown("**Matched-filter SNRs below carry this calibration uncertainty:** " + ", ".join(
    f"{PASS_NAMES[a]} ±{50 * (k['match_68'][1] - k['match_68'][0]) / k['match']:.0f}%"
    for a, k in KAPPA.items())))
show_figure(fig, "noise_check")

## Step 5 · Per-shot SNR: current narrow filter vs matched filter

For every shot, three numbers are computed:

- **Old metric:** what the single-shot notebooks report. The largest |Z| within ±8 ms of the PMT peak, divided by the RMS of both quadratures more than 20 ms away. Pure noise gives about 2.4.
- **Narrow, at t_c:** the same 0.5 ms filter, read at t_c, projected onto the signal phase, divided by its calibrated noise.
- **Matched, at t_c:** the Gaussian filter of the transit width, read and normalized the same way.

The last two use identical definitions, so their ratio is the **real gain from the filter**. The old metric is shown for continuity with the logbook only.

In [ ]:
per_shot = []
for d in signal_shots:
    acq, tr = d["acq"], d["tr"]
    old = analyze_shot(d["shot"], d["ref"], n_periods=2, cloud_half_width_s=0.008,
                       background_guard_s=0.012, switch_guard_s=0.001, harmonics=(1,))
    old_r = old["results"][("PDA100A2", 1)]
    x_narrow = cse.project(d["z_narrow_tc"], PHASE[acq])
    x_match = cse.project(d["z_match_tc"], PHASE[acq])
    d["noise_match_true"] = tr["noise_match"] * KAPPA[acq]["match"]
    per_shot.append({
        "file": d["file"], "pass": PASS_NAMES[acq],
        "old metric": old_r["peak_amp"] / old_r["noise_rms"],
        "narrow amp (mV)": 1e3 * x_narrow,
        "narrow SNR": x_narrow / (tr["noise_narrow"] * KAPPA[acq]["narrow"]),
        "matched amp (mV)": 1e3 * x_match,
        "matched SNR": x_match / d["noise_match_true"],
        "matched noise (mV)": 1e3 * d["noise_match_true"],
    })
per_shot = pd.DataFrame(per_shot)
summary5 = per_shot.groupby("pass")[["old metric", "narrow SNR", "matched SNR"]].agg(["mean", "std"])
summary5[("gain", "matched/narrow")] = summary5[("matched SNR", "mean")] / summary5[("narrow SNR", "mean")]
display(summary5.round(2))

fig, axes = plt.subplots(len(PHASE), 1, layout="constrained", squeeze=False)
for ax, acq in zip(axes[:, 0], PHASE):
    group = per_shot[per_shot["pass"] == PASS_NAMES[acq]]
    index = np.arange(len(group))
    ax.plot(index, group["old metric"], "s", color="0.5", label="old metric (max |Z|, both quadratures)")
    ax.plot(index, group["narrow SNR"], "o", color="C0", label="narrow filter at t_c")
    ax.plot(index, group["matched SNR"], "o", color="C3", label="matched filter at t_c")
    ax.set_title(f"{PASS_NAMES[acq]}: per-shot SNR", fontdict=title_font)
    ax.set_ylabel("SNR", fontdict=label_font, labelpad=12)
    ax.set_ylim(bottom=0)
axes[-1, 0].set_xlabel("shot (in loading order)", fontdict=label_font, labelpad=20)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside lower center", ncols=3, frameon=False, prop=legend_font)
show_figure(fig, "per_shot_snr")

## Step 6 · Is the transit width really the best filter width?

Theory, for a Gaussian pulse of width s and a Gaussian filter of width w in white noise:

$$\mathrm{SNR}(w) \propto \frac{s\,\sqrt{w}}{\sqrt{s^2 + w^2}}.$$

The peak is at **w = s**, and the curve is flat near it: a filter 30% too narrow or too wide loses only a few percent. Below, the measured mean SNR at each width uses noise calibrated on the no-launch shots *at that width*. The theory curve is scaled to the measured maximum. Measured points falling below the curve at large widths are the extra slow noise showing up.

In [ ]:
scan_rows = []
for acq in PHASE:
    for width in WIDTH_SCAN_S:
        kappa, n_cal, _ = calibrate_noise(acq, width_s=width)
        snrs = []
        for d in (s for s in signal_shots if s["acq"] == acq):
            tr = cse.shot_traces(d["shot"], d["ref"], d["pmt"]["center_s"], width)
            x = cse.project(cse.value_at(tr["t"], tr["z_match"], d["pmt"]["center_s"]), PHASE[acq])
            snrs.append(x / (tr["noise_match"] * kappa))
        scan_rows.append({"pass": PASS_NAMES[acq], "width (ms)": 1e3 * width, "kappa": kappa,
                          "calibration readings": n_cal, "mean SNR": float(np.mean(snrs))})
width_scan = pd.DataFrame(scan_rows)
display(width_scan.pivot(index="width (ms)", columns="pass", values=["mean SNR", "kappa"]).round(2))

fig, axes = plt.subplots(1, len(PHASE), layout="constrained", squeeze=False)
for ax, acq in zip(axes[0], PHASE):
    group = width_scan[width_scan["pass"] == PASS_NAMES[acq]]
    s_ms = 1e3 * SIGMA_MATCH[acq]
    w = np.linspace(0.05, group["width (ms)"].max(), 300)
    theory = s_ms * np.sqrt(w) / np.sqrt(s_ms ** 2 + w ** 2)
    ax.plot(w, theory / theory.max() * group["mean SNR"].max(), color="0.4", label="theory (scaled)")
    ax.plot(group["width (ms)"], group["mean SNR"], "o-", color="C3", label="measured")
    ax.axvline(s_ms, color="C2", ls="--", label=f"PMT transit σ = {s_ms:.2f} ms")
    ax.axvline(0.12, color="C0", ls=":", label="≈ current narrow filter")
    ax.set_xscale("log")
    ax.set_title(f"{PASS_NAMES[acq]}: SNR vs filter width", fontdict=title_font)
    ax.set_xlabel("Gaussian filter σ (ms)", fontdict=label_font, labelpad=20)
    ax.set_ylabel("mean per-shot SNR", fontdict=label_font, labelpad=12)
# The PMT-width line differs per panel, so each panel keeps its own small legend in a corner
# that stays empty (low SNR at the narrowest filters, upper left).
for ax in axes[0]:
    ax.legend(prop=legend_font, loc="upper left")
show_figure(fig, "width_scan")

## Step 7 · Averaging shots, aligned on the PMT

Two ways to average N shots' matched-filter traces X_signal(t):

- **Unaligned:** on the shared clock (time after drop). Shot-to-shot variation in the arrival time smears the average.
- **Aligned:** each shot is shifted so its own PMT-defined t_c sits at zero first. This is the same as reading every shot at its own t_c.

Here the arrival-time jitter (Step 2) is small compared with the transit width, so alignment should matter little. Most of the gain comes from the matched filter itself.

Two uncertainties are reported for the averaged amplitude:

- **Noise-limited:** the calibrated noise divided by √N. This is how well the *detection* resolves the average.
- **Shot-to-shot scatter:** the standard deviation of the per-shot amplitudes divided by √N. This includes any real shot-to-shot change in the signal itself. If it is much larger than the noise-limited value, the detection is no longer the limit; something in the experiment varies from shot to shot.

**What varies is not settled.** On 2026-10-14 the PMT fluorescence did *not* follow the PDA amplitude, so the scatter is not simply atom number. It was anti-correlated with the slow probe-power swing; see the logbook.

In [ ]:
avg_rows = []
fig, axes = plt.subplots(2, len(PHASE), layout="constrained", squeeze=False)
for col, acq in enumerate(PHASE):
    group = [d for d in signal_shots if d["acq"] == acq]
    s = SIGMA_MATCH[acq]
    tau = np.linspace(-3.5 * s, 3.5 * s, 281)            # time relative to the cloud centre
    t_mean = float(np.mean([d["pmt"]["center_s"] for d in group]))
    aligned, unaligned, narrow_aligned = [], [], []
    for d in group:
        tr = d["tr"]
        ok_m = tr["valid_match"]
        x_m = cse.project(tr["z_match"], PHASE[acq])
        x_n = cse.project(tr["z_narrow"], PHASE[acq])
        # np.interp outside the valid region would extrapolate edge values; mark those as NaN.
        def sample(trace, ok, times):
            out = np.interp(times, tr["t"], trace)
            out[(times < tr["t"][ok][0]) | (times > tr["t"][ok][-1])] = np.nan
            return out
        aligned.append(sample(x_m, ok_m, d["pmt"]["center_s"] + tau))
        unaligned.append(sample(x_m, ok_m, t_mean + tau))
        narrow_aligned.append(sample(x_n, tr["valid_narrow"], d["pmt"]["center_s"] + tau))
    # Average only where every shot has valid data; near the window edges only some shots
    # contribute, and a mean over a few shots there would look like a real feature.
    aligned, unaligned, narrow_aligned = (np.mean(np.array(a), axis=0)
                                          for a in (aligned, unaligned, narrow_aligned))

    amps = np.array([cse.project(d["z_match_tc"], PHASE[acq]) for d in group])
    noise = np.array([d["noise_match_true"] for d in group])
    n = len(group)
    noise_of_mean = float(np.sqrt(np.mean(noise ** 2)) / np.sqrt(n))
    scatter_of_mean = float(np.std(amps, ddof=1) / np.sqrt(n))
    avg_rows.append({"pass": PASS_NAMES[acq], "shots": n,
                     "mean amplitude (mV)": 1e3 * amps.mean(),
                     "noise-limited error (mV)": 1e3 * noise_of_mean,
                     "shot-scatter error (mV)": 1e3 * scatter_of_mean,
                     "SNR of average (noise-limited)": amps.mean() / noise_of_mean,
                     "SNR of average (incl. atom scatter)": amps.mean() / scatter_of_mean,
                     "aligned peak (mV)": 1e3 * np.nanmax(aligned),
                     "unaligned peak (mV)": 1e3 * np.nanmax(unaligned)})

    ax = axes[0, col]
    ax.plot(1e3 * tau, 1e3 * narrow_aligned, color="0.6", lw=0.8, label="narrow filter, aligned")
    ax.plot(1e3 * tau, 1e3 * unaligned, color="C0", lw=1.6, label="matched, unaligned")
    ax.plot(1e3 * tau, 1e3 * aligned, color="C3", lw=1.6, label="matched, PMT-aligned")
    ax.axhline(0, color="0.4", lw=0.6)
    ax.set_title(f"{PASS_NAMES[acq]}: average of {n} shots", fontdict=title_font)
    ax.set_xlabel("time from cloud centre (ms)", fontdict=label_font, labelpad=20)
    ax.set_ylabel("signal amplitude (mV)", fontdict=label_font, labelpad=12)

    # Cumulative SNR of the running mean, in loading order, against the sqrt(N) expectation.
    counts_n = np.arange(1, n + 1)
    running_mean = np.cumsum(amps) / counts_n
    running_noise = np.sqrt(np.cumsum(noise ** 2) / counts_n) / np.sqrt(counts_n)
    ax = axes[1, col]
    ax.plot(counts_n, running_mean / running_noise, "o-", color="C3", ms=3, label="running average")
    ax.plot(counts_n, np.mean(amps / noise) * np.sqrt(counts_n), color="0.4", ls="--",
            label="mean single-shot SNR × √N")
    ax.set_xlabel("number of shots averaged", fontdict=label_font, labelpad=20)
    ax.set_ylabel("SNR of the average", fontdict=label_font, labelpad=12)
# Shared legends outside the panels: traces on top, running SNR at the bottom.
fig.legend(*axes[0, 0].get_legend_handles_labels(), loc="outside upper center", ncols=3,
           frameon=False, prop=legend_font)
fig.legend(*axes[1, 0].get_legend_handles_labels(), loc="outside lower center", ncols=2,
           frameon=False, prop=legend_font)
display(pd.DataFrame(avg_rows).set_index("pass").round(3))
show_figure(fig, "averaging")

## Step 8 · One shot, three ways: raw, narrow filter, matched filter

This step shows what the filters do to a single shot. Choose it with `SHOT_TO_VIEW`: a file name from the Step 5 table, or `None` for the first returning shot. Three panels share the time axis (ms after drop):

1. **Raw photodiode voltage**, with its mean over the modulated window removed. What you see is the probe's own slow drift and the modulation's residual amplitude modulation. The absorption is a small fraction of a percent of the light and is invisible here.
2. **Demodulated, no filter (grey).** The same mixing with the reference as the lock-in, with the background removed and projected onto the signal phase, *before any smoothing*. Every sample still carries the full broadband noise, plus a ripple at 2·f_m left over from mixing. **Narrow filter (blue):** the same signal after the lock-in's 0.5 ms triangular filter.
3. **Narrow vs matched.** Blue is the narrow filter again; red is the Gaussian matched filter with the transit width. The shaded bands are each filter's ±1σ calibrated noise (Step 4). The dots mark the values read at t_c, the PMT-defined cloud centre, and those values give each filter's SNR.

The matched trace is lower and broader than the narrow one. It averages over the transit, which reduces the peak by about 1/√2, but the noise band shrinks far more. That trade is the SNR gain.

In [ ]:
# ---- CHOOSE THE SHOT: a file name from the Step 5 table, or None for the first returning shot ----
SHOT_TO_VIEW = None
# ---------------------------------------------------------------------------------------------------
if SHOT_TO_VIEW is None:
    view = next(d for d in signal_shots if d["acq"] == "1")
else:
    view = next(d for d in signal_shots if d["file"] == SHOT_TO_VIEW)
acq, tr, shot = view["acq"], view["tr"], view["shot"]
phase, t, t_c = PHASE[acq], tr["t"], view["pmt"]["center_s"]

def after_drop_ms(times):
    """Analysis-local seconds -> milliseconds after drop_time, as in the single-shot notebooks."""
    return 1e3 * (np.asarray(times) + shot["acquisition_after_drop_s"])

def only_valid(trace, ok):
    """Hide samples whose filter window runs past the modulated interval (edge artifacts)."""
    out = np.array(trace, dtype=float)
    out[~ok] = np.nan
    return out

enabled = (t >= shot["enable_start_s"]) & (t <= shot["enable_stop_s"])

# (1) Raw photodiode voltage, mean over the modulated window removed, in mV.
v_ac_mv = 1e3 * (shot["pda_v"] - shot["pda_v"][enabled].mean())

# (2) Demodulated but unfiltered: the lock-in's mixing step only, background removed, projected
#     onto this pass's signal phase. Shown only where the modulation is on. Without a filter it is
#     dominated by mixing products at f_m and 2*f_m (from the probe's slow drift and residual
#     amplitude modulation) plus broadband noise, so it gets its own panel and scale.
_, z_raw = cse.demodulate(shot, view["ref"])
x_raw_mv = 1e3 * only_valid(cse.project(z_raw - tr["z_bg"], phase), enabled)

# (3) The two filtered versions of the same signal, each shown only where its filter is valid.
x_narrow_mv = 1e3 * only_valid(cse.project(tr["z_narrow"], phase), tr["valid_narrow"])
x_match_mv = 1e3 * only_valid(cse.project(tr["z_match"], phase), tr["valid_match"])
noise_narrow_mv = 1e3 * tr["noise_narrow"] * KAPPA[acq]["narrow"]   # calibrated in Step 4
noise_match_mv = 1e3 * view["noise_match_true"]
at_tc_narrow = 1e3 * cse.project(view["z_narrow_tc"], phase)
at_tc_match = 1e3 * cse.project(view["z_match_tc"], phase)
raw_spread_mv = float(np.nanstd(x_raw_mv[tr["background"]]))

display(pd.DataFrame({
    "value at t_c (mV)": [np.nan, at_tc_narrow, at_tc_match],
    "noise, 1σ (mV)": [raw_spread_mv, noise_narrow_mv, noise_match_mv],
    "SNR at t_c": [np.nan, at_tc_narrow / noise_narrow_mv, at_tc_match / noise_match_mv],
}, index=["raw demodulated (per sample)", "narrow filter (0.5 ms)",
          f"matched filter (σ = {1e3 * SIGMA_MATCH[acq]:.2f} ms)"]).round(3))

fig, axes = plt.subplots(3, 1, sharex=True, layout="constrained")
for ax in axes:
    ax.axvline(after_drop_ms(t_c), color="C2", ls="--", lw=1)
    ax.axvspan(after_drop_ms(t_c - SIGMA_MATCH[acq]), after_drop_ms(t_c + SIGMA_MATCH[acq]),
               color="C2", alpha=0.08)
axes[0].plot(after_drop_ms(t), v_ac_mv, lw=0.4, color="0.3")
axes[0].set_ylabel("PDA − mean (mV)", fontdict=label_font, labelpad=12)
axes[0].set_title("1 · raw photodiode voltage (absorption not visible at this scale)",
                  fontdict=title_font)

axes[1].plot(after_drop_ms(t), x_raw_mv, lw=0.3, color="0.55")
axes[1].axhline(0, color="0.4", lw=0.6)
axes[1].set_ylabel("signal (mV)", fontdict=label_font, labelpad=12)
axes[1].set_title(f"2 · mixed with the reference, no filter: spread {raw_spread_mv:.0f} mV rms "
                  f"vs signal ≈ {max(at_tc_narrow, at_tc_match):.1f} mV", fontdict=title_font)

axes[2].fill_between(after_drop_ms(t), -noise_narrow_mv, noise_narrow_mv, color="C0", alpha=0.12,
                     label="±1σ noise, narrow")
axes[2].fill_between(after_drop_ms(t), -noise_match_mv, noise_match_mv, color="C3", alpha=0.25,
                     label="±1σ noise, matched")
axes[2].plot(after_drop_ms(t), x_narrow_mv, lw=0.8, color="C0", label="narrow filter (0.5 ms)")
axes[2].plot(after_drop_ms(t), x_match_mv, lw=2.0, color="C3",
             label=f"matched filter (σ = {1e3 * SIGMA_MATCH[acq]:.1f} ms)")
axes[2].plot(after_drop_ms(t_c), at_tc_narrow, "o", color="C0", ms=6)
axes[2].plot(after_drop_ms(t_c), at_tc_match, "o", color="C3", ms=7)
axes[2].axhline(0, color="0.4", lw=0.6)
axes[2].set_ylabel("signal (mV)", fontdict=label_font, labelpad=12)
axes[2].set_xlabel("Time after drop_time (ms)", fontdict=label_font, labelpad=20)
axes[2].set_title(f"3 · read at t_c: SNR {at_tc_narrow / noise_narrow_mv:.1f} (narrow) → "
                  f"{at_tc_match / noise_match_mv:.1f} (matched)", fontdict=title_font)
axes[2].set_xlim(after_drop_ms(shot["enable_start_s"]), after_drop_ms(shot["enable_stop_s"]))
fig.suptitle(f"{view['file']} ({PASS_NAMES[acq]}); green: PMT cloud centre ± transit σ",
             **title_font)
# Legend for panel 3 underneath the figure, so no trace is covered.
fig.legend(*axes[2].get_legend_handles_labels(), loc="outside lower center", ncols=2,
           frameon=False, prop=legend_font)
EXPORT_SIZES_INCHES["one_shot"] = (14, 12)
show_figure(fig, "one_shot")

## Summary and caveats

- **The matched filter is a pure analysis gain.** It uses the same data, and its width comes from the PMT, not tuned on the photodiode. The width scan should show the optimum near the PMT transit width.
- **The calibrated noise matters:** the white-noise prediction underestimates the wide-filter noise by κ. Without the no-launch calibration, the SNRs would be about 20–30% too high.
- **The matched filter gives up time resolution.** It averages over the whole transit, so it measures *how much* absorption there is, not fine detail in its shape. For timing or shape, use the single-shot notebooks, or look at Step 8.
- **Settings must match.** If the probe power, gain or modulation change, take new no-launch shots for the calibration, since the noise level depends on them.
- When the shot-scatter error is much larger than the noise-limited error, the detection is no longer the limit: the signal itself varies from shot to shot. On 2026-10-14 that variation did not follow the PMT, so it is not simply atom number. The probe-power swing is the leading suspect.

In [ ]:
# Save the per-shot table, the width scan and the averages next to the figures.
if SAVE_OUTPUTS:
    per_shot.to_csv(OUTPUT_DIR / "per_shot_snr.csv", index=False)
    width_scan.to_csv(OUTPUT_DIR / "width_scan.csv", index=False)
    pd.DataFrame(avg_rows).to_csv(OUTPUT_DIR / "averages.csv", index=False)
    print(f"Saved tables and figures to {OUTPUT_DIR}")